# Exercise 03: Regression Trees, Random Forest and Gradient Boosting – Concrete Strength

**Dataset:** 1'030 concrete samples tested in the laboratory (Yeh, 1998). For each sample the mixture (kg per m³ of
concrete), the age at testing (days) and the measured compressive strength (MPa) are known.
Source: [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength)
/ [Kaggle](https://www.kaggle.com/datasets/maajdl/yeh-concret-data).

| variable | meaning |
|---|---|
| `cement`, `slag`, `flyash`, `water`, `superplasticizer`, `coarseaggregate`, `fineaggregate` | components in kg/m³ |
| `age` | age of the sample at testing (days, 1–365) |
| `csMPa` | compressive strength (MPa) – **target** |

**Business question:** A construction company wants to predict the strength of a new concrete mixture *before* waiting
28 days for the laboratory test. Strength depends on the components in a strongly non-linear way and with interactions
(e.g. water-to-cement ratio, hardening over time) – a typical use case for tree-based models.

**Slides:** Part 03 «Regression models» – regression trees, pruning, hyperparameters, bagging, random forest, gradient
boosting, model comparison.

**After this exercise you can ...**
- find the best split of a regression tree by hand (SSE criterion),
- fit, visualise and prune a regression tree,
- explain and apply bagging, random forests (incl. out-of-bag error, permutation importance) and gradient boosting,
- compare models fairly with cross-validation.

## Concept

**Regression tree.** The tree cuts the feature space into boxes and predicts one constant per box: the mean of the
training targets in that box. A split (feature $j$, threshold $s$) is chosen greedily so that the total squared error of
the two children is minimal:

$$\min_{j,\,s}\; \Big[\sum_{x_i \in R_1}(y_i-\bar y_{R_1})^2 + \sum_{x_i \in R_2}(y_i-\bar y_{R_2})^2\Big]$$

**Cost-complexity pruning.** $R_\alpha(T) = \text{SSE}(T) + \alpha\,|T|$, where $|T|$ is the number of leaves. Larger
$\alpha$ → smaller tree. In scikit-learn: `ccp_alpha`, chosen by cross-validation.

**Why averaging helps.** For $B$ models with variance $\sigma^2$ and pairwise correlation $\rho$:

$$\operatorname{Var}\Big(\frac{1}{B}\sum_{b=1}^B \hat f_b\Big) = \rho\,\sigma^2 + \frac{1-\rho}{B}\,\sigma^2$$

- **Bagging:** fit deep trees on $B$ bootstrap samples and average. Each tree does not see ≈ 37 % of the observations
  (*out-of-bag*, OOB) → internal error estimate.
- **Random forest:** bagging + only $m$ of the $p$ features are considered at each split → de-correlated trees
  (smaller $\rho$) → lower variance.
- **Gradient boosting:** small trees are added one after the other; each new tree $h_m$ fits the current residuals
  (squared loss): $\hat f_m(x) = \hat f_{m-1}(x) + \nu\, h_m(x)$ with learning rate $\nu$.

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import tree
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: Calculations by hand

**a) Best split of a regression tree.** Six samples of the same mixture were tested at different ages:

| age $x$ [days] | 3 | 7 | 14 | 28 | 56 | 90 |
|---|---|---|---|---|---|---|
| strength $y$ [MPa] | 15 | 22 | 30 | 38 | 42 | 44 |

Compute the SSE of the root node. Then compute, for each of the five possible thresholds (midpoints between neighbouring
ages), the means and the SSE of the two child nodes. Which split does the tree choose? What are the leaf predictions?

**b) Variance of an average.** 100 trees have variance $\sigma^2 = 4$ each. Compute the variance of their average for
$\rho = 0.6$ (bagged trees, similar to each other) and for $\rho = 0.2$ (random forest). What happens for
$B \to \infty$?

**c) One boosting step.** The current prediction for a sample is 30 MPa, the observed strength is 40 MPa. The next tree
predicts the residual exactly. What is the new prediction with learning rate $\nu = 0.1$? How many such steps are needed
until the remaining residual is below 5 MPa?

**d)** Verify your results with Python.

In [ ]:
# a) Best split
x = np.array([3, 7, 14, 28, 56, 90])
y = np.array([15, 22, 30, 38, 42, 44])

def sse(v):
    return np.sum((v - v.mean())**2)

print(f'Root: mean = {y.mean():.2f}, SSE = {sse(y):.2f}')
rows = []
for s in (x[:-1] + x[1:]) / 2:
    left, right = y[x <= s], y[x > s]
    rows.append({'split': f'x <= {s}', 'mean left': left.mean(), 'mean right': right.mean(),
                 'SSE left': sse(left), 'SSE right': sse(right), 'SSE total': sse(left) + sse(right)})
print(pd.DataFrame(rows).round(2).to_string(index=False))

# Check with scikit-learn: a tree with one split (a "stump")
stump = DecisionTreeRegressor(max_depth=1).fit(x.reshape(-1, 1), y)
print(tree.export_text(stump, feature_names=['age']))

# b) Variance of an average
sigma2, B = 4, 100
for rho in [0.6, 0.2]:
    print(f'rho = {rho}: Var = {rho*sigma2 + (1-rho)/B*sigma2:.3f}  (limit for B -> inf: {rho*sigma2})')

# c) Boosting steps
y_obs, pred, nu, steps = 40, 30, 0.1, 0
while y_obs - pred >= 5:
    pred = pred + nu * (y_obs - pred)
    steps += 1
    if steps == 1:
        print('Prediction after 1 step:', pred)
print(f'Steps until residual < 5: {steps} (prediction {pred:.2f})')

**Solution:**

a) Root: $\bar y = 191/6 \approx 31.83$, SSE $\approx 672.83$.

| split | left node | mean | right node | mean | SSE left + SSE right |
|---|---|---|---|---|---|
| $x \le 5$ | {15} | 15 | {22, 30, 38, 42, 44} | 35.2 | 0 + 332.8 = 332.8 |
| $x \le 10.5$ | {15, 22} | 18.5 | {30, 38, 42, 44} | 38.5 | 24.5 + 115 = 139.5 |
| $x \le 21$ | {15, 22, 30} | 22.33 | {38, 42, 44} | 41.33 | 112.67 + 18.67 = **131.33** |
| $x \le 42$ | {15, 22, 30, 38} | 26.25 | {42, 44} | 43 | 296.75 + 2 = 298.75 |
| $x \le 73$ | {15, …, 42} | 29.4 | {44} | 44 | 495.2 + 0 = 495.2 |

The tree chooses **age ≤ 21** (SSE drops from 672.8 to 131.3, −80 %). Leaf predictions: 22.33 MPa (young samples) and
41.33 MPa (old samples).

b) $\rho = 0.6$: $0.6\cdot 4 + 0.4/100\cdot 4 = 2.416$; $\rho = 0.2$: $0.2\cdot 4 + 0.8/100\cdot 4 = 0.832$.
For $B \to \infty$ only the shared part $\rho\sigma^2$ remains (2.4 resp. 0.8). More trees cannot remove it – this is
why the random forest de-correlates the trees.

c) Residual 10 → new prediction $30 + 0.1\cdot 10 = 31$. Each step removes 10 % of the remaining residual:
$10\cdot 0.9^m < 5$ → $m = 7$ steps (residual $10\cdot 0.9^7 \approx 4.78$). A small learning rate needs many trees but
makes the model less prone to overfitting.

## Exercise 2: Import and explore the data

a) Import the data, check for missing values and duplicates (remove duplicates), and show descriptive statistics.

b) Create scatter plots of `csMPa` against `cement`, `water` and `age`. Which relationships are clearly non-linear?

c) Split the data into training (80 %) and test set (20 %) with `random_state=42`.

In [ ]:
# a) Import the data
df = pd.read_csv('./Data/concrete_data.csv', sep=',')

print('Missing values:', df.isna().sum().sum())
print('Duplicates:    ', df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print(df.shape)
print(df.describe().round(1).T)

# b) Scatter plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, col in zip(axes, ['cement', 'water', 'age']):
    ax.scatter(df[col], df['csMPa'], s=8, alpha=0.5)
    ax.set_xlabel(col)
    ax.grid(alpha=0.3)
axes[0].set_ylabel('Compressive strength [MPa]')
plt.show()

# c) Train/test split
X = df.drop(columns='csMPa')
y = df['csMPa']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print('Training set:', X_train.shape, ' Test set:', X_test.shape)

**Solution:**

a) No missing values; 25 duplicated rows are removed (1'005 samples remain).

b) Strength increases roughly linearly with `cement` and decreases with `water`, with a lot of scatter. The relationship
with `age` is clearly **non-linear**: strength rises steeply during the first 28 days and then levels off (hardening
curve). The age values are discrete (1, 3, 7, 14, 28, 56, 90, … days) – a step function, as produced by a tree, is a
natural fit.

## Exercise 3: Baseline – linear regression

Fit a multiple linear regression with all features on the training set and compute RMSE and $R^2$ on the test set. This
is the baseline that the tree-based models have to beat.

In [ ]:
lin_reg = LinearRegression().fit(X_train, y_train)
y_pred_lin = lin_reg.predict(X_test)
print('Linear regression - test RMSE:', round(root_mean_squared_error(y_test, y_pred_lin), 2),
      ' test R2:', round(r2_score(y_test, y_pred_lin), 3))

## Exercise 4: A small regression tree

a) Fit a `DecisionTreeRegressor(max_depth=3, random_state=42)` on the training set.

b) Visualise the tree with `tree.plot_tree()` and interpret the first two levels. Which features are used?

c) Compute RMSE and $R^2$ on the test set. How does the small tree compare to the linear regression?

In [ ]:
# a) Fit the tree
reg_tree = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_train, y_train)

# b) Plot the tree
plt.figure(figsize=(18, 7))
tree.plot_tree(reg_tree, feature_names=list(X_train.columns), filled=True,
               rounded=True, fontsize=9, precision=1)
plt.show()

# c) Test RMSE and R2
y_pred_tree = reg_tree.predict(X_test)
print('Tree (depth 3) - test RMSE:', round(root_mean_squared_error(y_test, y_pred_tree), 2),
      ' test R2:', round(r2_score(y_test, y_pred_tree), 3))

**Solution:**

The root split is **age ≤ 21 days** – the same threshold as in the hand calculation of Exercise 1a (hardening before
vs. after about three weeks). On the second level both branches split on **cement** (≈ 350 kg/m³); further down
**water**, **superplasticizer** and cement again are used. The tree reads like rules of thumb: "older than 21 days, more
than 352 kg cement and little water → ≈ 62 MPa". With only 8 leaves the tree predicts only 8 different values; its test
RMSE (≈ 11.3 MPa) is about the same as for the linear regression (≈ 11.2 MPa) – easy to interpret, but high bias.

## Exercise 5: Tree complexity and pruning

a) For `max_depth` = 1, 2, …, 20 compute the RMSE on the training set and the 5-fold CV RMSE (on the training set).
Plot both curves. Where does the tree start to overfit?

b) Alternatively, grow a full tree and prune it: compute the candidate values with
`DecisionTreeRegressor(random_state=42).cost_complexity_pruning_path(X_train, y_train).ccp_alphas` and choose `ccp_alpha`
with `GridSearchCV` (5-fold CV, `scoring='neg_root_mean_squared_error'`). How many leaves does the best pruned tree have?
What is its test RMSE?

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# a) Training vs. CV error for different depths
depths = range(1, 21)
rmse_train, rmse_cv = [], []
for d in depths:
    m = DecisionTreeRegressor(max_depth=d, random_state=42)
    rmse_cv.append(-cross_val_score(m, X_train, y_train, cv=cv,
                                    scoring='neg_root_mean_squared_error').mean())
    rmse_train.append(root_mean_squared_error(y_train, m.fit(X_train, y_train).predict(X_train)))

plt.figure(figsize=(7, 4))
plt.plot(depths, rmse_train, marker='o', label='training error')
plt.plot(depths, rmse_cv, marker='o', label='5-fold CV error')
plt.xlabel('max_depth')
plt.ylabel('RMSE [MPa]')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print('Best depth by CV:', depths[int(np.argmin(rmse_cv))], ' CV RMSE:', round(min(rmse_cv), 2))

# b) Cost-complexity pruning
alphas = DecisionTreeRegressor(random_state=42).cost_complexity_pruning_path(X_train, y_train).ccp_alphas
grid_tree = GridSearchCV(DecisionTreeRegressor(random_state=42),
                         {'ccp_alpha': alphas},
                         cv=cv, scoring='neg_root_mean_squared_error')
grid_tree.fit(X_train, y_train)
pruned_tree = grid_tree.best_estimator_
print('Number of candidate alphas:', len(alphas))
print('Best ccp_alpha:', round(grid_tree.best_params_['ccp_alpha'], 4),
      ' CV RMSE:', round(-grid_tree.best_score_, 2))
print('Leaves: full tree =', DecisionTreeRegressor(random_state=42).fit(X_train, y_train).get_n_leaves(),
      ' pruned tree =', pruned_tree.get_n_leaves())
print('Pruned tree - test RMSE:', round(root_mean_squared_error(y_test, pruned_tree.predict(X_test)), 2))

**Solution:**

a) The training error drops to almost 0 for deep trees (each leaf contains only one or very few samples). The CV error
decreases until a depth of about 10–12 (CV RMSE ≈ 7.1 MPa) and then stays flat – the additional splits only learn noise:
training and CV error drift apart (high variance).

b) Pruning removes the leaves that reduce the training error only a little: the best pruned tree has about 160 leaves
instead of about 760 for the full tree, with a CV RMSE (≈ 7.3 MPa) comparable to the best depth-limited tree. Its test
RMSE (≈ 6.4 MPa) is clearly better than the linear regression (≈ 11.2 MPa).

## Exercise 6: Random forest

a) Fit random forests with `n_estimators` = 1, 5, 10, 25, 50, 100, 200, 400 (`oob_score=True`, `random_state=42`,
`n_jobs=-1`). The OOB predictions are stored in `oob_prediction_`. Plot the OOB RMSE against the number of trees. When do
the gains plateau?

b) Compare `max_features` = 1.0 (all features, i.e. bagging), `'sqrt'` and 0.5 with 5-fold CV (300 trees).

c) For the best forest, compute the test RMSE. Compare the impurity-based feature importance (`feature_importances_`)
with the **permutation importance** on the test set.

In [ ]:
# a) Number of trees vs. OOB error
n_trees = [1, 5, 10, 25, 50, 100, 200, 400]
oob_rmse = []
for n in n_trees:
    rf = RandomForestRegressor(n_estimators=n, oob_score=True, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    pred_oob = rf.oob_prediction_
    ok = ~np.isnan(pred_oob)            # with very few trees, some samples are never out-of-bag
    oob_rmse.append(root_mean_squared_error(y_train[ok], pred_oob[ok]))

plt.figure(figsize=(7, 4))
plt.plot(n_trees, oob_rmse, marker='o')
plt.xscale('log')
plt.xlabel('n_estimators (log scale)')
plt.ylabel('OOB RMSE [MPa]')
plt.title('Random forest: OOB error vs. number of trees')
plt.grid(alpha=0.3)
plt.show()
print(pd.DataFrame({'n_estimators': n_trees, 'OOB RMSE': np.round(oob_rmse, 2)}).to_string(index=False))

# b) max_features
for mf in [1.0, 'sqrt', 0.5]:
    rf = RandomForestRegressor(n_estimators=300, max_features=mf, random_state=42, n_jobs=-1)
    score = -cross_val_score(rf, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error').mean()
    print(f'max_features = {mf}: CV RMSE = {score:.2f}')

In [ ]:
# c) Test RMSE and feature importance (use the best max_features from b)
rf_best = RandomForestRegressor(n_estimators=300, max_features=0.5, random_state=42, n_jobs=-1)
rf_best.fit(X_train, y_train)
print('Random forest - test RMSE:', round(root_mean_squared_error(y_test, rf_best.predict(X_test)), 2),
      ' test R2:', round(r2_score(y_test, rf_best.predict(X_test)), 3))

perm = permutation_importance(rf_best, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
imp = pd.DataFrame({'impurity-based (MDI)': rf_best.feature_importances_,
                    'permutation (test set)': perm.importances_mean},
                   index=X_train.columns).sort_values('permutation (test set)')
print(imp.round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
imp['impurity-based (MDI)'].plot(kind='barh', ax=axes[0], color='grey', title='Impurity-based importance')
imp['permutation (test set)'].plot(kind='barh', ax=axes[1], color='darkred',
                                   title='Permutation importance (increase in error)')
plt.show()

**Solution:**

a) A single tree has a high OOB error; the error drops quickly with the first 25 trees and **plateaus** at about
5.1 MPa after 50–100 trees. More trees never hurt the accuracy, they only cost computing time.

b) `max_features=0.5` (4 of 8 features per split) is slightly better than plain bagging (1.0), because the trees become
less correlated. `'sqrt'` (only 2 of 8 features) is worse here: with so few candidates, the important features (age,
cement) are often not available at a split, which increases the bias of each tree. The best $m$ depends on the data →
tune it. (Note: scikit-learn uses all features by default for regression.)

c) The random forest reaches a test RMSE of about 5.1 MPa ($R^2 \approx 0.91$) – less than half of the linear
regression error. Both importance
measures agree that **age** and **cement** are the most important features, followed by water. Impurity-based importance
is computed on the training data and can favour features with many split points; permutation importance measures how
much the test error increases if a feature is shuffled and is therefore the more reliable measure.

## Exercise 7: Gradient boosting

a) Fit `GradientBoostingRegressor(n_estimators=1000, max_depth=3, random_state=42)` with learning rates
$\nu$ = 0.01, 0.1 and 1.0. Use `staged_predict(X_test)` to compute the test RMSE after each boosting iteration and plot
the three curves.

*Note: we look at the test curve here only for illustration. In practice, `learning_rate` and `n_estimators` are tuned
with cross-validation (see Exercise 8).*

b) Describe the influence of the learning rate. What happens with $\nu = 1.0$?

In [ ]:
# a) Test error per iteration for different learning rates
plt.figure(figsize=(8, 4))
for lr in [0.01, 0.1, 1.0]:
    gb = GradientBoostingRegressor(n_estimators=1000, max_depth=3, learning_rate=lr, random_state=42)
    gb.fit(X_train, y_train)
    rmse_staged = [root_mean_squared_error(y_test, p) for p in gb.staged_predict(X_test)]
    plt.plot(range(1, 1001), rmse_staged, label=f'learning rate {lr}')
    print(f'learning rate {lr}: min test RMSE = {min(rmse_staged):.2f} after {int(np.argmin(rmse_staged)) + 1} trees,'
          f' after 1000 trees = {rmse_staged[-1]:.2f}')
plt.ylim(0, 20)
plt.xlabel('Number of trees M')
plt.ylabel('Test RMSE [MPa]')
plt.title('Gradient boosting: test error vs. number of trees')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Solution:**

- $\nu = 0.01$: very small steps – the error decreases slowly and is still decreasing after 1'000 trees (≈ 5.5 MPa).
- $\nu = 0.1$: the error decreases quickly and reaches the lowest value (≈ 4.1 MPa); this is the default and usually a
  good compromise.
- $\nu = 1.0$: each tree corrects the full residual. The error drops very fast at the beginning, reaches its minimum
  (≈ 5.1 MPa) after about 100 trees and then **rises again** – the model fits the noise of the training data
  (overfitting).

Learning rate and number of trees must be chosen **together**: a smaller $\nu$ needs a larger $M$.

## Exercise 8: Fair model comparison

Compare the following models with **5-fold CV RMSE on the training set** and summarise the results in a table:
linear regression, pruned tree (best `ccp_alpha` from Exercise 5), random forest (best setting from Exercise 6) and
gradient boosting (`learning_rate=0.1`, `n_estimators=500`, `max_depth=3`). Then evaluate the best model **once** on the
test set. Which model would you recommend to the construction company, and what are the drawbacks?

In [ ]:
models = {
    'Linear regression': LinearRegression(),
    'Pruned tree':       DecisionTreeRegressor(ccp_alpha=grid_tree.best_params_['ccp_alpha'], random_state=42),
    'Random forest':     RandomForestRegressor(n_estimators=300, max_features=0.5, random_state=42, n_jobs=-1),
    'Gradient boosting': GradientBoostingRegressor(learning_rate=0.1, n_estimators=500, max_depth=3, random_state=42),
}

rows = []
for name, m in models.items():
    scores = -cross_val_score(m, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error')
    rows.append({'model': name, 'CV RMSE (mean)': scores.mean(), 'CV RMSE (std)': scores.std()})
comparison = pd.DataFrame(rows).sort_values('CV RMSE (mean)')
print(comparison.round(2).to_string(index=False))

best_name = comparison.iloc[0]['model']
best_model = models[best_name].fit(X_train, y_train)
print(f'\nBest model by CV: {best_name}')
print('Test RMSE:', round(root_mean_squared_error(y_test, best_model.predict(X_test)), 2),
      ' test R2:', round(r2_score(y_test, best_model.predict(X_test)), 3))

**Solution:**

The ensembles are clearly best: gradient boosting (CV RMSE ≈ 4.7 MPa) before the random forest (≈ 5.3 MPa), followed by
the pruned tree (≈ 7.3 MPa) and the linear regression (≈ 10.2 MPa). On the test set gradient boosting reaches an RMSE of
≈ 4.3 MPa and $R^2 \approx 0.94$.

Recommendation: gradient boosting (or random forest) for the prediction. Drawbacks: the models are black boxes (only
feature importance, no simple formula or rules), they cannot extrapolate beyond the range of the training data (e.g.
new materials, higher cement contents), and gradient boosting needs careful tuning of learning rate, number and depth of
trees. If interpretability is required (e.g. for a standard), a small pruned tree or a linear model with suitable
transformations (e.g. log(age), water/cement ratio) can be a reasonable compromise.

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')